# glcuda Wave 129 - T4 fused specialization compiler gate
Compiler-resource gate only. No parity or throughput claim is made by this notebook.


In [ ]:
import base64
import hashlib
import json
from pathlib import Path
import re
import subprocess
import traceback
import zipfile

BUILD = "wave129-fused-specialization-v1"
PTX_SHA256 = "80cdd36f64ce8873679d1a75ea0e01ad7525177a26f4ea69fe0d37737be4d06b"
PTX_B64 = "..."
ROOT = Path("/kaggle/working/wave129")
RESULTS = ROOT / "results"
PTX = RESULTS / "glcuda_sm75_wave129.ptx"
CUBIN = ROOT / "wave129.cubin"
ARCHIVE = Path("/kaggle/working/glcuda-t4-wave129-fused-specialization-results.zip")
RESULTS.mkdir(parents=True, exist_ok=True)


def run(cmd):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, text=True, capture_output=True, timeout=7200)
    print(p.stdout[-20000:], flush=True)
    print(p.stderr[-20000:], flush=True)
    if p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p


def archive():
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
    print("ARCHIVE", ARCHIVE, digest, flush=True)


phase = "bootstrap"
try:
    ptx = base64.b64decode(PTX_B64)
    if hashlib.sha256(ptx).hexdigest() != PTX_SHA256:
        raise RuntimeError("embedded PTX SHA-256 mismatch")
    PTX.write_bytes(ptx)
    (RESULTS / "source.json").write_text(
        json.dumps({"build": BUILD, "ptx_sha256": PTX_SHA256, "ptx_bytes": len(ptx)}, indent=2),
        encoding="utf-8",
    )

    phase = "gpu"
    smi = run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
    (RESULTS / "nvidia-smi.log").write_text(smi.stdout + smi.stderr, encoding="utf-8")
    if "Tesla T4" not in smi.stdout:
        raise RuntimeError("Wave 129 requires a Tesla T4")

    phase = "compiler-resource"
    ptxas = Path("/usr/local/cuda/bin/ptxas")
    if not ptxas.is_file():
        raise RuntimeError(f"ptxas unavailable: {ptxas}")
    built = run([ptxas, "-v", "-arch=sm_75", PTX, "-o", CUBIN])
    log = built.stdout + "\n--- STDERR ---\n" + built.stderr
    (RESULTS / "ptxas-wave129.log").write_text(log, encoding="utf-8")

    def amount(pattern):
        found = re.search(pattern, log)
        if not found:
            raise RuntimeError(f"missing ptxas field: {pattern}")
        return int(found.group(1))

    resource = {
        "registers_per_thread": amount(r"Used\s+(\d+)\s+registers"),
        "barriers": amount(r"used\s+(\d+)\s+barriers"),
        "static_shared_bytes": amount(r"(\d+)\s+bytes smem"),
        "stack_frame_bytes": amount(r"(\d+)\s+bytes stack frame"),
        "spill_store_bytes": amount(r"(\d+)\s+bytes spill stores"),
        "spill_load_bytes": amount(r"(\d+)\s+bytes spill loads"),
    }
    passed = (
        resource["registers_per_thread"] <= 80
        and resource["barriers"] == 1
        and resource["static_shared_bytes"] == 16384
        and resource["stack_frame_bytes"] == 0
        and resource["spill_store_bytes"] == 0
        and resource["spill_load_bytes"] == 0
    )
    summary = {
        "build": BUILD,
        "gpu": "Tesla T4",
        "resource": resource,
        "resource_gate_passed": passed,
        "production_dispatch_wired": False,
        "parity_run": False,
        "production_timing_run": False,
        "target_15000_tps_achieved": False,
    }
    (RESULTS / "wave129-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("WAVE129_RESULT", json.dumps(summary, indent=2), flush=True)
    if not passed:
        raise RuntimeError(f"Wave 129 compiler resource gate failed: {resource}")
except Exception:
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "traceback": traceback.format_exc()}, indent=2),
        encoding="utf-8",
    )
    archive()
    raise

archive()